In [8]:
import os 
import json
import torch
import pickle
import pandas as pd 
import numpy as np 
from attrdict import AttrDict
from bertviz import model_view
from transformers import BertForSequenceClassification
from transformers import BertConfig, BertTokenizer, BertModel

In [9]:
default_path = os.getcwd()
base_model = 'C:/Users/lamda/Desktop/LAMDA_git/EmoDep/base-model'
config_path = 'C:/Users/lamda/Desktop/LAMDA_git/EmoDep/config'
model_path = "C:/Users/lamda/Desktop/LAMDA_git/EmoDep/model/emodep/"
config_file = "bert-base.json"
save_path = os.path.join(default_path, './')

In [10]:
tokenizer = BertTokenizer.from_pretrained(os.path.join(base_model, 'bert-base'), model_max_length=128)
config = BertConfig.from_pretrained(os.path.join(base_model, 'bert-base', 'bert_config.json'), num_labels=9, output_hidden_states=True, output_attentions=True)
model = BertForSequenceClassification.from_pretrained(os.path.join(base_model, 'bert-base'), config=config)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at C:/Users/lamda/Desktop/LAMDA_git/EmoDep/base-model\bert-base and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
with open(os.path.join(config_path, 'training_config.json')) as f:
    training_config = AttrDict(json.load(f))

training_config.pad = 'max_length'
training_config.device = torch.device("cuda") if torch.cuda.is_available() else "cpu"
model.to(training_config.device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [12]:
config.max_position_embeddings = 128

In [13]:
model_name = os.path.join(model_path, 'bert_emodep_e5.pt')

In [14]:
model.load_state_dict(torch.load(model_name, map_location=torch.device('cpu')))
model.to(training_config.device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [15]:
input_text = "I haven't had an appetite since last week"  
# input_text = "I can not sleep well these days"
# input_text = "I am so depressed these days"
inputs = tokenizer.encode(input_text, return_tensors='pt').to(training_config.device)

In [16]:
len(inputs[0])

12

In [17]:
outputs = model(inputs)  # Run model
attention = outputs[-1]  # Retrieve attention from model outputs
tokens = tokenizer.convert_ids_to_tokens(inputs[0])  # Convert input ids to token strings
model_view(attention, tokens)  # Display model view

<IPython.core.display.Javascript object>

In [18]:
from bertviz import head_view

head_view(attention, tokens)

<IPython.core.display.Javascript object>